In [1]:
import geopandas as gpd
import numpy as np
from libpysal import graph

In [2]:
def graph_to_edge_set(g, unique_ids):
    """
    Convert a libpysal Graph to upper-triangle (i < j) integer index pairs
    matching the sparse matrix row/column indices.
    """
    id_to_int = {uid: idx for idx, uid in enumerate(unique_ids)}
    adj = g.adjacency.reset_index()

    i_series = adj['focal'].map(id_to_int)
    j_series = adj['neighbor'].map(id_to_int)

    valid = i_series.notna() & j_series.notna()
    i_arr = i_series[valid].astype(int).values
    j_arr = j_series[valid].astype(int).values

    mask = i_arr != j_arr
    i_arr, j_arr = i_arr[mask], j_arr[mask]

    upper_i = np.minimum(i_arr, j_arr)
    upper_j = np.maximum(i_arr, j_arr)

    return list(set(zip(upper_i.tolist(), upper_j.tolist())))
    
def get_deletable_edges(W_sparse):
    """Edges where both endpoints have degree > 1 (no isolates created)."""
    W_sparse = W_sparse.tocsr()
    deletable = []
    for i in range(W_sparse.shape[0]):
        degree_i = W_sparse.getrow(i).nnz - (1 if W_sparse[i, i] != 0 else 0)
        if degree_i > 1:
            row = W_sparse.getrow(i).tocoo()
            for j in row.col:
                if i < j:
                    degree_j = W_sparse.getcol(j).nnz - (1 if W_sparse[j, j] != 0 else 0)
                    if degree_j > 1:
                        deletable.append((i, j))
    return deletable


def get_deletable_edges_mst(W_sparse, max_components):
    """Edges safe to delete without exceeding max_components."""
    G = nx.from_scipy_sparse_array(W_sparse)
    current_components = nx.number_connected_components(G)
    all_edges = set(G.edges())
    non_bridges = all_edges - set(nx.bridges(G))

    if current_components >= max_components:
        return list(non_bridges)
    return list(all_edges)

In [5]:
def build_g_borders(gdf, g_rook, region_col):
    g_block = graph.Graph.build_block_contiguity(gdf[region_col])
    g_borders = g_rook.difference(g_block) # rook-block
    g_remaining = g_rook.difference(g_borders) # rook-borders
    return g_borders, g_remaining

def build_g_region(gdf, g_rook, region_col, region_id=None):
    if region_id is None:
        region_id = np.random.choice(gdf[region_col].unique())
    region_ids = gdf[gdf[region_col] == region_id].index
    edges_df = g_rook.adjacency.reset_index()
    mask = edges_df['focal'].isin(region_ids) | edges_df['neighbor'].isin(region_ids) # at least edge node within selected region
    edges_region = edges_df[mask]
    g_region = graph.Graph.from_adjacency(edges_region, 'focal', 'neighbor', 'weight')
    g_remaining = g_rook.difference(g_region)
    return g_region, g_remaining

In [6]:
def remove_one_edge(W_sparse, method, max_components=None, candidate_edges=None):
    if method == "no_isolates":
        deletable = get_deletable_edges(W_sparse)
    else:
        deletable = get_deletable_edges_mst(W_sparse, max_components=max_components)

    if candidate_edges is not None:
        candidate_set = set(candidate_edges)
        deletable = [(i, j) for i, j in deletable if (i, j) in candidate_set]

    if len(deletable) == 0:
        return W_sparse.copy(), False

    i, j = deletable[np.random.randint(len(deletable))]
    W_new = W_sparse.tolil()
    W_new[i, j] = 0
    W_new[j, i] = 0
    return W_new.tocsr(), True

def run_corruption_loop(W_sparse, method, max_components=None, candidate_edges=None):
    # remove edges one at a time until no deletable edges are left
    W = W_sparse.copy()
    corrupted_graphs = []
    while True:
        W, success = remove_one_edge(
            W, method=method,
            max_components=max_components,
            candidate_edges=candidate_edges,
        )
        if not success:
            break
        corrupted_graphs.append(graph.Graph.from_sparse(W))
    return corrupted_graphs

In [7]:
sizes = [400,100,25]
shapes = ["square","pent","hex"]

In [9]:
n_corrupt = 10

In [10]:
for shape in shapes:
    for size in sizes:
        g_true = graph.read_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
        W_sparse = g_true.to_W().sparse
        for n in range(n_corrupt):
            W = W_sparse.copy()
            iteration = 0
            while True:
                W, success = remove_one_edge(W, method='no_isolates')
                if not success:
                    break
                iteration += 1
                corrupted_g = graph.Graph.from_sparse(W)
                corrupted_g.to_parquet(
                    f"graphs/{shape}/size_{size}/no_isolates/random/g_missing{iteration}_run{n}.parquet")

KeyboardInterrupt: 

In [12]:
for shape in shapes:
    for size in [100,400]:
        g_true = graph.read_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
        W_sparse = g_true.to_W().sparse
        for n in range(n_corrupt):
            W = W_sparse.copy()
            iteration = 0
            while True:
                W, success = remove_one_edge(W, method='mst',max_components = 1)
                if not success:
                    break
                iteration += 1
                corrupted_g = graph.Graph.from_sparse(W)
                corrupted_g.to_parquet(
                    f"graphs/{shape}/size_{size}/one_component/random/g_missing{iteration}_run{n}.parquet")

In [13]:
for shape in shapes:
    for size in [100, 400]:
        gdf = gpd.read_parquet(f"data/regionalization/gdf_{shape}_{size}.parquet")
        g_true = graph.read_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
        W_sparse = g_true.to_W().sparse
        for n in range(n_corrupt):
            region_col = f"zone_id_{n}"

            # borders
            g_borders, _ = build_g_borders(gdf, g_true, region_col=region_col)
            border_edges = graph_to_edge_set(g_borders, g_true.unique_ids)
            for condition, kwargs in [
                ("one_component", dict(method='mst', max_components=1)),
                ("no_isolates",   dict(method='no_isolates')),
            ]:
                corrupted_graphs = run_corruption_loop(
                    W_sparse, candidate_edges=border_edges, **kwargs
                )
                for i, corrupted_g in enumerate(corrupted_graphs, start=1):
                    corrupted_g.to_parquet(
                        f"graphs/{shape}/size_{size}/{condition}/borders/g_missing{i}_run{n}.parquet"
                    )

            # local
            region_id = np.random.choice(gdf[region_col].unique())
            g_region, _ = build_g_region(gdf, g_true, region_col=region_col, region_id=region_id)
            local_edges = graph_to_edge_set(g_region, g_true.unique_ids)
            for condition, kwargs in [
                ("one_component", dict(method='mst', max_components=1)),
                ("no_isolates",   dict(method='no_isolates')),
            ]:
                corrupted_graphs = run_corruption_loop(
                    W_sparse, candidate_edges=local_edges, **kwargs
                )
                for i, corrupted_g in enumerate(corrupted_graphs, start=1):
                    corrupted_g.to_parquet(
                        f"graphs/{shape}/size_{size}/{condition}/local/g_missing{i}_run{n}.parquet"
                    )